# Sensitivity concepts for variance-based analysis

**Leif Rune Hellevik**
NTNU / University of Michigan

## Setup

In [1]:
# --- cell: install_chaospy ---
# @title Install chaospy (Colab-friendly)

try:
    import chaospy as cp
    import numpoly
    import numpy as np
    print("chaospy er allerede installert.")
except ImportError:
    # Installer chaospy fra PyPI. Dette drar inn numpoly automatisk.
    %pip install chaospy==4.3.21 --no-cache-dir
    import chaospy as cp
    import numpoly
    import numpy as np

print("numpy  :", np.__version__)
print("numpoly:", numpoly.__version__)
print("chaospy:", cp.__version__)

chaospy er allerede installert.
numpy  : 2.2.6
numpoly: 1.3.6
chaospy: 4.3.20


In [2]:
# --- cell: repo_setup ---
# @title Repo sync and environment setup

import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REMOTE = "https://github.com/lrhgit/uqsa2025.git"
REPO_PATH_COLAB = Path("/content/uqsa2025")

if IN_COLAB:
    if not REPO_PATH_COLAB.exists():
        print("Cloning repository...")
        subprocess.run(
            ["git", "clone", REMOTE, str(REPO_PATH_COLAB)],
            check=True
        )
    else:
        print("Updating existing repository...")
        subprocess.run(
            ["git", "-C", str(REPO_PATH_COLAB), "pull"],
            check=True
        )
    os.chdir(REPO_PATH_COLAB)

# --- Find repo root (works locally + in Colab) ---
cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in [cwd] + list(cwd.parents) if (p / ".git").exists()),
    cwd
)

PY_SRC = repo_root / "python_source"
if PY_SRC.exists() and str(PY_SRC) not in sys.path:
    sys.path.insert(0, str(PY_SRC))

print("CWD:", Path.cwd())
print("repo_root:", repo_root)
print("python_source exists:", PY_SRC.exists())
print("python_source in sys.path:", str(PY_SRC) in sys.path)

CWD: /Users/leifh/git/uqsa2025
repo_root: /Users/leifh/git/uqsa2025
python_source exists: True
python_source in sys.path: True


In [3]:
# --- cell: layout_and_numpy_patch ---
# @title Layout fix, imports, and NumPy compatibility patch

import warnings
warnings.filterwarnings("ignore")

from IPython.display import HTML

HTML("""
<style>
div.cell.code_cell, div.output {
    max-width: 100% !important;
}
</style>
""")

import numpy as np
import matplotlib.pyplot as plt
import chaospy as cp
import numpoly
import pandas as pd


# Pretty-print helpers (used across notebooks)
from pretty_printing import (
    section_title,
    pretty_table,
    pretty_print_sobol_mc,
    sensitivity_table,
)


# --- NumPy reshape compatibility patch for numpoly ---
_old_reshape = np.reshape

def _reshape_compat(a, *args, **kwargs):
    newshape = None
    if "newshape" in kwargs:
        newshape = kwargs.pop("newshape")
    if "shape" in kwargs and newshape is None:
        newshape = kwargs.pop("shape")
    if newshape is not None:
        return _old_reshape(a, newshape, *args, **kwargs)
    return _old_reshape(a, *args, **kwargs)

np.reshape = _reshape_compat
print("✓ numpy.reshape patched for numpoly compatibility")

✓ numpy.reshape patched for numpoly compatibility


In [4]:
# --- cell: load_problem_helpers ---
import numpy as np
import chaospy as cp

from sensitivity_examples_nonlinear import (
    analytic_sensitivity_coefficients,
)

## From intuition to formal language

In the interactive G* exploration, we observed that some factors matter more than others, and that first-order and total-effect indices may differ.

Variance-based sensitivity analysis provides a formal way to quantify how uncertainty in model inputs contributes to uncertainty in model outputs.

We now connect that intuition to the interpretation of Sobol sensitivity indices.


## A simple linear model

To build intuition, we start with a simple additive model:

$$
Y = \sum_{i=1}^{r} \Omega_i Z_i
$$

where the uncertain inputs $Z_i$ are independent random variables, and the coefficients $\Omega_i$ determine how strongly each factor influences the output.

We assume

$$
Z_i \sim N(0,\sigma_{Z_i}), \qquad i=1,\ldots,r
$$

so that the output is also normally distributed, with

$$
\bar y = \sum_{i=1}^{r} \Omega_i \bar z_i,
\qquad
\sigma_Y =
\sqrt{
\sum_{i=1}^{r} \Omega_i^2 \sigma_{Z_i}^2
}
$$

To make the effects visually clear, we order the uncertainties:

$$
\sigma_{Z_1} < \sigma_{Z_2} < \cdots < \sigma_{Z_r}
$$

In [5]:
def linear_model(Z, omega):
    """
    Z     : input samples, shape (Ns, Nrv)
    omega : coefficients, shape (Nrv,)
    """
    return Z @ omega

In [6]:
Nrv = 4  # number of random variables

mu = np.zeros(Nrv)  # expected value
sigma = np.array([1.0, 2.0, 3.0, 4.0]) #std
omega = np.ones(Nrv) * 2.0  # weights


In [7]:
Ns = 500

# Independent Gaussian input factors
pdfs = [
    cp.Normal(mu_i, sigma_i)
    for mu_i, sigma_i in zip(mu, sigma)
]

jpdf = cp.J(*pdfs)

# Monte Carlo input sample matrix
Z = jpdf.sample(Ns).T   # (Ns, Nrv)

# Model evaluations
Y = linear_model(Z, omega)

After defining the input distributions, we draw a Monte Carlo sample and evaluate the linear model for each sample. This gives paired values of each input factor \(Z_i\) and the corresponding model output \(Y\), which we can inspect using scatterplots.

## Scatterplots as a first diagnostic

Use the sliders to change the coefficients \(\Omega_i\), and inspect how the scatterplots change.


In [8]:
   # Interactive scatterplot demo for the linear model
from scatter_demo import scatter_demo

scatter_ui = scatter_demo(Z)
display(scatter_ui)

**Reflection**

Based on the scatterplots:

- Which input appears most influential?
- Which input appears least influential?
- Can you rank the inputs by apparent influence?
- Does the ranking depend only on \(\Omega_i\), or also on the spread of \(Z_i\)?

The scatterplots suggest that influence depends both on the model coefficient \(\Omega_i\) and on the variability of the input \(Z_i\). This motivates sensitivity measures that account for input scaling.

## Normalized derivatives

Raw derivatives measure how strongly the output changes with respect to an input:

$$
\frac{\partial Y}{\partial Z_i}
$$

However, raw derivatives depend on scaling and units, making comparisons between parameters difficult.

A normalized sensitivity measure is obtained by scaling both input and output with their standard deviations:

$$
S_{Z_i}^{\sigma}
=
\frac{\sigma_{Z_i}}{\sigma_Y}
\frac{\partial Y}{\partial Z_i}
\tag{10}
$$

For the linear model,

$$
Y = \sum_{i=1}^{r} \Omega_i Z_i,
$$

the normalized sensitivities become

$$
\left(S_{Z_i}^{\sigma}\right)^2
=
\left(
\frac{\Omega_i \sigma_{Z_i}}
{\sigma_Y}
\right)^2
\tag{11}
$$

This has several attractive properties:

- both the model coefficients $\Omega_i$ and the input variability $\sigma_{Z_i}$ influence the sensitivity
- the sensitivities are dimensionless
- they are directly comparable across parameters
- for additive linear models:

$$
\sum_i \left(S_{Z_i}^{\sigma}\right)^2 = 1
$$

The normalized sensitivities therefore provide a natural ranking of parameter importance for the linear model.

In [9]:
# Analytical sensitivity ranking
display(section_title("Analytical sensitivity ranking"))

sensitivity = (omega * sigma) ** 2
sensitivity /= np.sum(sensitivity)

sensitivity_table(
    sensitivity,
    [f"Z_{i}" for i in range(1, Nrv + 1)],
    column_name="normalized sensitivity",
)

,normalized sensitivity
Z_1,0.033
Z_2,0.133
Z_3,0.300
Z_4,0.533


## From conditional variances to Sobol indices

The scatterplots suggest that influential factors produce visible structure or patterns in the model output $Y$.

To make this idea more quantitative, we may divide the range of an input factor $Z_i$ into slices and examine how the distribution of $Y$ changes from slice to slice.

The interactive plot below illustrates this idea.

In [10]:
from conditional_slices import conditional_slices_interactive

ui = conditional_slices_interactive(Z.T, omega, jpdf)
ui

## From normalized derivatives to Sobol indices

For a very thin slice, we effectively keep $Z_i$ fixed while averaging over all remaining uncertain inputs. This corresponds to the conditional expectation

$$
E_{Z_{\sim i}}(Y \mid Z_i),
$$

where $Z_{\sim i}$ denotes all variables except $Z_i$.

If this conditional expectation varies strongly with $Z_i$, the factor is influential. A natural quantitative measure is therefore the conditional variance

$$
V_{Z_i}\!\left(E_{Z_{\sim i}}(Y \mid Z_i)\right).
$$

The first-order Sobol sensitivity index is this conditional variance normalized by the total output variance:

$$
S_i
=
\frac{
V_{Z_i}\!\left(E_{Z_{\sim i}}(Y \mid Z_i)\right)
}{
V(Y)
}.
$$

A large value of $S_i$ indicates that $Z_i$ explains a large fraction of the output variability.

### Connection to normalized derivatives for additive linear models

For the additive linear model,

$$
Y = \sum_{i=1}^r \Omega_i Z_i,
$$

there are no interaction terms. The first-order Sobol indices therefore account for all output variance, and reduce to

$$
S_i
=
\frac{\Omega_i^2 \sigma_{Z_i}^2}{\sigma_Y^2}.
$$

Using the normalized derivative sensitivity measure introduced earlier,

$$
S_{Z_i}^{\sigma}
=
\frac{\sigma_{Z_i}}{\sigma_Y}
\frac{\partial Y}{\partial Z_i},
$$

and noting that

$$
\frac{\partial Y}{\partial Z_i} = \Omega_i,
$$

we obtain

$$
S_i
=
\left(
S_{Z_i}^{\sigma}
\right)^2.
$$

Thus, for additive linear models, the first-order Sobol indices are identical to the squared normalized derivative sensitivities.